## This is for PDFs

In [ ]:
from pathlib import Path
import pymupdf
import pytesseract
from pdf2image import convert_from_path


def extract_pdf_pages(pdf_path: str, min_chars_per_page: int = 30) -> list[dict]:
    """
    Returns page-level records:
    {
        'page_number': 1,
        'text': '...',
        'extraction_method': 'text' or 'ocr'
    }
    """
    pdf_path = Path(pdf_path)
    doc = pymupdf.open(pdf_path)

    pages = []

    for page_index, page in enumerate(doc):
        text = page.get_text("text").strip()

        pages.append({
            "page_number": page_index + 1,
            "text": text,
            "extraction_method": "text"
        })

    doc.close()

    # OCR only pages with very little extractable text
    sparse_pages = [
        x["page_number"]
        for x in pages
        if len(x["text"].strip()) < min_chars_per_page
    ]

    if sparse_pages:
        images = convert_from_path(
            str(pdf_path),
            dpi=300,
            first_page=min(sparse_pages),
            last_page=max(sparse_pages)
        )

        image_lookup = {
            page_no: image
            for page_no, image in zip(
                range(min(sparse_pages), max(sparse_pages) + 1),
                images
            )
        }

        for page_record in pages:
            page_no = page_record["page_number"]

            if page_no in sparse_pages:
                ocr_text = pytesseract.image_to_string(
                    image_lookup[page_no],
                    lang="eng"
                ).strip()

                page_record["text"] = ocr_text
                page_record["extraction_method"] = "ocr"

    return pages

## This is for Docx

In [ ]:
from docx import Document


def extract_docx_text(docx_path: str) -> list[dict]:
    document = Document(docx_path)
    sections = []

    paragraph_text = "\n".join(
        p.text.strip()
        for p in document.paragraphs
        if p.text.strip()
    )

    if paragraph_text:
        sections.append({
            "page_number": None,
            "text": paragraph_text,
            "extraction_method": "docx_paragraphs"
        })

    for table_index, table in enumerate(document.tables, start=1):
        rows = []

        for row in table.rows:
            values = [cell.text.strip() for cell in row.cells]
            rows.append(" | ".join(values))

        table_text = "\n".join(rows).strip()

        if table_text:
            sections.append({
                "page_number": None,
                "text": f"[Table {table_index}]\n{table_text}",
                "extraction_method": "docx_table"
            })

    return sections

### Step 2: Retrieve candidate passages
Do not run an LLM over every full document. First find likely passages with rules.

A good initial approach is a sliding window around lines that contain drawdown-related terms. For every hit, take several lines before and after it. This preserves the sentence that may state “three Business Days.”



In [ ]:
import re


DRAW_DOWN_TERMS = [
    r"\bdrawdown\b",
    r"\bdraw down\b",
    r"\butilisation\b",
    r"\butilization\b",
    r"\butilisation request\b",
    r"\butilization request\b",
    r"\bborrowing request\b",
    r"\bloan request\b",
    r"\badvance request\b",
    r"\bnotice of drawdown\b",
]

NOTICE_TERMS = [
    r"\bnotice\b",
    r"\bprior written notice\b",
    r"\bnot later than\b",
    r"\bat least\b",
    r"\bbusiness days?\b",
    r"\bworking days?\b",
]


def find_candidate_passages(
    text: str,
    lines_before: int = 4,
    lines_after: int = 8
) -> list[str]:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    candidates = []

    for i, line in enumerate(lines):
        line_lower = line.lower()

        has_drawdown_term = any(
            re.search(pattern, line_lower, flags=re.IGNORECASE)
            for pattern in DRAW_DOWN_TERMS
        )

        if has_drawdown_term:
            start = max(0, i - lines_before)
            end = min(len(lines), i + lines_after + 1)

            passage = "\n".join(lines[start:end])

            has_notice_context = any(
                re.search(pattern, passage, flags=re.IGNORECASE)
                for pattern in NOTICE_TERMS
            )

            if has_notice_context:
                candidates.append(passage)

    return list(dict.fromkeys(candidates))

You can also score candidates. Give a higher score when the excerpt contains both “Utilisation Request” and “Business Days,” then rank the best evidence for each deal.

In [ ]:
def score_candidate(text: str) -> int:
    text_lower = text.lower()
    score = 0

    if any(x in text_lower for x in ["drawdown", "draw down", "utilisation", "utilization"]):
        score += 3

    if "request" in text_lower:
        score += 2

    if "notice" in text_lower:
        score += 2

    if "business day" in text_lower or "working day" in text_lower:
        score += 4

    if "not later than" in text_lower or "at least" in text_lower:
        score += 2

    return score

### Step 3: Extract notice days deterministically

Many clauses follow predictable patterns. First use regex extraction because it is fast, reproducible, and cheap.

In [ ]:
import re


BUSINESS_DAY_PATTERNS = [
    r"(\d+)\s*\(\s*\w+\s*\)\s+Business\s+Days?",
    r"(\d+)\s+Business\s+Days?",
    r"(\w+)\s+Business\s+Days?",
    r"not\s+later\s+than\s+(\d+)\s+Business\s+Days?",
    r"at\s+least\s+(\d+)\s+Business\s+Days?",
]


NUMBER_WORDS = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
}


def extract_business_days(text: str) -> list[int]:
    results = []

    for pattern in BUSINESS_DAY_PATTERNS:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        for match in matches:
            value = match.strip().lower()

            if value.isdigit():
                results.append(int(value))
            elif value in NUMBER_WORDS:
                results.append(NUMBER_WORDS[value])

    return sorted(set(results))

### Step 4: Add an LLM only for semantic validation
Rules retrieve the candidate text; an LLM can then classify and extract the clause from a limited, evidence-rich passage.

You can use:
- An approved enterprise LLM/API if documents may leave your environment
- Azure OpenAI / OpenAI only if internal policy permits
- A local LLM through Ollama if confidentiality prevents external transfer
- A human-review queue for low-confidence cases

For confidential loan documentation, check your organisation’s data-handling policy before uploading documents or excerpts to any external model provider.

Use a constrained JSON prompt. The model should be allowed to say "not_found" or "unclear"; never force it to invent a number.



SYSTEM_PROMPT = """
You are a contract-clause extraction assistant.

Analyse only the supplied document excerpt. Do not use external knowledge.
Determine whether the excerpt contains a clause requiring a borrower or another
party to submit a drawdown, utilisation, borrowing, advance, or loan request
before funds are made available.

If present, extract the required prior notice period in Business Days or
Working Days.

Return valid JSON only using this schema:
{
  "drawdown_notice_clause": "yes" | "no" | "unclear",
  "notice_period_business_days": integer | null,
  "notice_deadline_description": string | null,
  "evidence_quote": string | null,
  "reason": string,
  "confidence": "high" | "medium" | "low"
}

Rules:
- Return a number only when the excerpt explicitly states it.
- Do not confuse payment deadlines, interest periods, conditions precedent,
  or post-drawdown timeframes with a prior drawdown-notice requirement.
- The evidence_quote must be copied exactly from the provided excerpt.
"""

Example Payload:

user_prompt = f"""
Document: {document_name}
Page: {page_number}

Excerpt:
{candidate_passage}
"""

A robust decision policy
Use a hierarchy:

1. High confidence
- A candidate contains drawdown-related language plus advance notice plus a stated number of business/working days.
- The extracted number and evidence agree between regex and LLM.

2. Medium confidence
- The LLM finds a valid drawdown-notice requirement, but the count is expressed unusually or spread across nearby clauses.
- Send to reviewer.
 
3. Low confidence / review required
- OCR quality is poor.
- The excerpt contains terms but no explicit notice obligation.
- The evidence is found in a template or an unrelated schedule.
- Multiple contradictory notice periods occur in amendments/restatements.

4. Not found
- No candidate appears after searching all available documents.
- Phrase this as “not found in processed documents,” not “does not exist.”

### Step 5: Create an auditable output
Your final output should be a table such as:


| Deal_ID | Clause found | Notice days | Document | Page | Evidence | Method | Confidence | Review status |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| DEAL-001 | Yes | 3 | Facility Agreement.pdf | 42 | “...delivered not later than three Business Days...” | Regex + LLM | High | Completed |
| DEAL-002 | Unclear | — | Loan Schedule.docx | — | “...notice shall be given...” | LLM | Medium | Human review |
| DEAL-003 | Not found | — | All processed documents | — | — | Keyword + semantic search | Low | Review if material |

Save the raw candidate passages too. A separate evidence table is useful because a single Deal_ID may have:

- Original facility agreement
- Restatement agreement
- Amendment letter
- Separate drawdown notice template
- Schedules with different facilities or tranches

A later amendment could supersede the number in the original agreement.

In [13]:
import pandas as pd


results_df = pd.DataFrame(results)

results_df.to_csv(
    "output/drawdown_clause_results.csv",
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(
    "output/drawdown_clause_results.xlsx",
    engine="xlsxwriter"
) as writer:
    results_df.to_excel(writer, sheet_name="Clause Results", index=False)

ModuleNotFoundError: No module named 'pandas'

#### A complete minimal pipeline
This example covers PDF and DOCX files, keeps page/source metadata, finds candidate clauses, and exports results. It does not include the LLM call, so you can test it safely first.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import fitz
from docx import Document


DRAW_DOWN_TERMS = [
    r"\bdrawdown\b",
    r"\bdraw down\b",
    r"\butilisation\b",
    r"\butilization\b",
    r"\butilisation request\b",
    r"\butilization request\b",
    r"\bborrowing request\b",
    r"\bloan request\b",
    r"\badvance request\b",
]

NOTICE_TERMS = [
    r"\bnotice\b",
    r"\bprior written notice\b",
    r"\bnot later than\b",
    r"\bat least\b",
    r"\bbusiness days?\b",
    r"\bworking days?\b",
]

NUMBER_WORDS = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
}


def extract_pdf_pages(file_path: str) -> list[dict]:
    doc = fitz.open(file_path)
    pages = []

    for page_index, page in enumerate(doc):
        pages.append({
            "page_number": page_index + 1,
            "text": page.get_text("text").strip(),
            "extraction_method": "pymupdf"
        })

    doc.close()
    return pages


def extract_docx_sections(file_path: str) -> list[dict]:
    document = Document(file_path)
    results = []

    paragraphs = "\n".join(
        p.text.strip()
        for p in document.paragraphs
        if p.text.strip()
    )

    if paragraphs:
        results.append({
            "page_number": None,
            "text": paragraphs,
            "extraction_method": "python-docx-paragraphs"
        })

    for table_index, table in enumerate(document.tables, start=1):
        table_rows = []

        for row in table.rows:
            table_rows.append(
                " | ".join(cell.text.strip() for cell in row.cells)
            )

        table_text = "\n".join(table_rows).strip()

        if table_text:
            results.append({
                "page_number": None,
                "text": f"[Table {table_index}]\n{table_text}",
                "extraction_method": "python-docx-table"
            })

    return results


def extract_document(file_path: str) -> list[dict]:
    extension = Path(file_path).suffix.lower()

    if extension == ".pdf":
        return extract_pdf_pages(file_path)

    if extension == ".docx":
        return extract_docx_sections(file_path)

    return []


def find_candidate_passages(text: str) -> list[str]:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    candidates = []

    for index, line in enumerate(lines):
        line_lower = line.lower()

        has_drawdown_term = any(
            re.search(pattern, line_lower, re.IGNORECASE)
            for pattern in DRAW_DOWN_TERMS
        )

        if not has_drawdown_term:
            continue

        start = max(0, index - 4)
        end = min(len(lines), index + 9)
        passage = "\n".join(lines[start:end])

        has_notice_term = any(
            re.search(pattern, passage, re.IGNORECASE)
            for pattern in NOTICE_TERMS
        )

        if has_notice_term:
            candidates.append(passage)

    return list(dict.fromkeys(candidates))


def extract_business_days(text: str) -> list[int]:
    patterns = [
        r"(\d+)\s*\(\s*\w+\s*\)\s+Business\s+Days?",
        r"(\d+)\s+Business\s+Days?",
        r"(\w+)\s+Business\s+Days?",
        r"(\d+)\s+Working\s+Days?",
        r"(\w+)\s+Working\s+Days?",
    ]

    values = []

    for pattern in patterns:
        for match in re.findall(pattern, text, re.IGNORECASE):
            match = match.lower().strip()

            if match.isdigit():
                values.append(int(match))
            elif match in NUMBER_WORDS:
                values.append(NUMBER_WORDS[match])

    return sorted(set(values))


def score_candidate(passage: str) -> int:
    text = passage.lower()
    score = 0

    if any(term in text for term in ["drawdown", "utilisation", "utilization", "borrowing request"]):
        score += 3

    if "notice" in text:
        score += 2

    if "business day" in text or "working day" in text:
        score += 4

    if "not later than" in text or "at least" in text:
        score += 2

    return score


deal_map = pd.read_csv("data/deal_document_map.csv")
output_rows = []

for _, record in deal_map.iterrows():
    deal_id = record["Deal_ID"]
    file_path = record["file_path"]
    document_type = record.get("document_type", None)

    try:
        units = extract_document(file_path)
    except Exception as exc:
        output_rows.append({
            "Deal_ID": deal_id,
            "file_path": file_path,
            "document_type": document_type,
            "page_number": None,
            "status": "extraction_error",
            "error": str(exc),
            "candidate_passage": None,
            "business_days_candidates": None,
            "candidate_score": None,
            "extraction_method": None
        })
        continue

    for unit in units:
        passages = find_candidate_passages(unit["text"])

        for passage in passages:
            days = extract_business_days(passage)

            output_rows.append({
                "Deal_ID": deal_id,
                "file_path": file_path,
                "document_type": document_type,
                "page_number": unit["page_number"],
                "status": "candidate_found",
                "error": None,
                "candidate_passage": passage,
                "business_days_candidates": ", ".join(map(str, days)) if days else None,
                "candidate_score": score_candidate(passage),
                "extraction_method": unit["extraction_method"]
            })

results_df = pd.DataFrame(output_rows)

results_df = results_df.sort_values(
    ["Deal_ID", "candidate_score"],
    ascending=[True, False]
)

results_df.to_excel(
    "output/drawdown_clause_candidates.xlsx",
    index=False
)

#### When to use embeddings or RAG
Add semantic search only after your baseline keyword pipeline is running.

It becomes valuable when the language varies widely, for example:
- “The Borrower shall submit a request for an Advance…”
- “A utilisation may be requested by delivery of a duly completed notice…”
- “No Loan may be made unless the Agent receives the request by…”
- “The funding request shall be submitted prior to the proposed funding date…”

In that case, use a local vector database—no Azure subscription required:
- ChromaDB for a straightforward local vector store
- FAISS for efficient local similarity search
- SQLite + FTS5 / BM25 for transparent lexical retrieval
- SentenceTransformers for local embeddings

For your project, I would use a hybrid retrieval design:

Candidate score = keyword score + BM25 score + semantic similarity score

Then retrieve the top 3–10 passages for each Deal ID and send only those passages to your clause-extraction step. The general RAG indexing/retrieval pattern can be run locally; it does not require Azure AI Search.

A good second-phase stack is:

`pip install sentence-transformers chromadb rank-bm25`